# 01. Mutability and aliasing

**Tier:** Deep. **Why you'd use this:** You pass a list into a function, the function appends, and the caller sees the change. Knowing what is mutable — and how copies work — stops that surprise from becoming a bug.

**After this lesson you can:** say which function calls the caller can see; tell a shallow copy from a deep one; protect a nested list.
About 30 minutes. **Uses:** [Objects, names, references](../01-objects/01-names-lesson.ipynb), [Arguments](../../course/05-functions/02-arguments-lesson.ipynb)


## Quick review

1. When do two names share one object?
2. How do `is` and `==` differ?

<details><summary>Answers</summary>

After assignment or when both are bound to the same literal/object. `is` asks identity; `==` asks equal value.
</details>


## The idea

A function parameter is another name for the object the caller passed. Python does not copy the argument, and it does not pass a hidden clone. This is call by sharing: the callee shares the object, and it has its own slot to point at that object.

Mutating the object is visible to every name that still points at it, including the caller's. Rebinding the parameter writes only the callee's slot. When the function returns, that slot is gone, and the caller still points where it did.

A copy makes a new container. A shallow copy copies the top container and reuses the inner objects. A deep copy walks the whole structure and copies those too.

<details><summary>Explain more</summary>

Sharing is why a method on a list argument shows up outside the function, and why `parameter = something_else` does not. The slot and the object are different writes. Copying chooses how many of the reachable objects get a new identity.

Step the two function calls in [Python Tutor](https://pythontutor.com/visualize.html). Watch the caller's arrow stay put while the parameter arrow moves, or stay aimed at one list while `append` changes that list.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

This is the same story as passing an object reference, including the part where assigning the parameter does not replace the caller's variable. There is no pass-by-reference that lets the callee reseat the caller's name.
</details>


## See it work

A grid of sensor rows is a list of lists. You want a private copy before a calibration writes into row 0. Copying only the outer list leaves the row objects shared, so the write lands in the original. `deepcopy` copies the rows too.


In [1]:
import copy


def private_grid(grid: list[list[int]]) -> list[list[int]]:
    # 1. Copy every level the calibration will write.
    return copy.deepcopy(grid)


grid = [[0, 0], [1, 1]]
private = private_grid(grid)
private[0].append(9)
print(grid)
print(private)
print(grid[0] is private[0])


[[0, 0], [1, 1]]
[[0, 0, 9], [1, 1]]
False


| step | grid[0] | private[0] | same row object? |
| --- | --- | --- | --- |
| after deepcopy | [0, 0] | [0, 0] | no |
| after append 9 | [0, 0] | [0, 0, 9] | no |

```
grid    ──► [ rowA, rowB ]
private ──► [ rowA', rowB' ]
```

A shallow copy would have drawn both outer lists pointing at rowA. The append would have changed `grid` too. The next cells measure that difference with `is`, which is stable, instead of printing raw `id` numbers that change every process.


## Check yourself

Answer these from the example above before you run the cell.


1. Why doesn't `readings = readings + [sample]` update the caller?
2. Why does `grid is shallow` stay false while `grid[0] is shallow[0]` stays true?
3. What would a naive deep copy do on `loop.append(loop)` without a memo of already-copied objects?

<details><summary>Answers</summary>

1. The addition allocates a new list and the assignment stores it in the callee's slot. The caller's slot is a different namespace entry and is not written.
2. The shallow copy allocated a new outer list, so the outer `is` is false. It stored the old row pointers inside that new list, so the row `is` is true.
3. It would follow the self-reference forever, copying a new list for every hop. The memo stops the walk when it sees an identity it has already copied.
</details>


In [2]:
def accumulate(readings: list[int], sample: int) -> None:
    readings.append(sample)


def rebind(readings: list[int], sample: int) -> None:
    readings = readings + [sample]


data = [1]
accumulate(data, 2)
print("after append", data)
rebind(data, 3)
print("after rebind", data)


after append [1, 2]
after rebind [1, 2]


<details><summary>Check your answers</summary>

- After `accumulate`, `data` is `[1, 2]`. `append` mutated the one list the caller and the parameter both point at.
- After `rebind`, `data` is still `[1, 2]`. `readings + [sample]` built a new list, and the assignment stored it only in the parameter. The caller never received that pointer.
</details>


## Why it works this way

The call `accumulate(data, 2)` creates a frame and initializes the slot `readings` with the pointer stored under `data`. No list is allocated. `append` asks that object to grow. The caller's slot still holds the same pointer, so the caller sees the new element. That is the whole of call by sharing: new slots, same objects, unless someone allocates.

`readings = readings + [sample]` does allocate. List addition builds a new list. The assignment then overwrites the parameter slot. The caller's slot is a different entry in a different namespace, the caller's frame, and nothing in that line writes it. Returning the new list would hand the pointer back. Discarding it drops the only reference, and the new list is garbage. People describe this as "Python is pass by value" or "pass by reference". Neither phrase is precise. The value that is copied into the parameter is the pointer. The object is shared. The caller's name is not.

A shallow copy allocates one new container and fills it with the pointers from the old one. For a list of ints, those pointers reach immutable objects, so the copy behaves like a private list: you can append to either side without the other noticing. For a list of lists, the inner lists are mutable objects reached by copied pointers. Time and extra memory are proportional to the length of the outer list, not to the size of the rows. `copy.copy` and `list(grid)` and `grid.copy()` and `grid[:]` are that same shallow copy.

`deepcopy` walks every object it can reach and allocates a counterpart, then fixes the pointers so the new structure points at the new objects. Time and memory follow the whole graph, not the top list. A cycle would make a naive walk infinite. `deepcopy` remembers objects it has already copied, so a list that contains itself becomes a new list that contains itself, not an endless loop. That memo is a dictionary from old identity to new object, which is why identity, not `==`, decides "have I copied this already".

The design reason for sharing at the call is cost and intention. Copying every argument would make passing a million-element list expensive and would hide mutation, so methods like `list.sort` could not update the caller's list. Sharing makes the cheap call the default and makes mutation explicit. Copying is available when you need isolation, and you choose the depth because a deep copy of a large graph is the expensive call.

The cells below print `is` for the outer list and for row 0, then show a self-referential list surviving `deepcopy`.


In [3]:
grid = [[0], [1]]
shallow = list(grid)
deep = copy.deepcopy(grid)
print("outer", grid is shallow, grid is deep)
print("row", grid[0] is shallow[0], grid[0] is deep[0])
shallow[0].append(9)
print("after shallow write", grid, shallow, deep)


outer False False
row True False
after shallow write [[0, 9], [1]] [[0, 9], [1]] [[0], [1]]


In [4]:
loop = [1]
loop.append(loop)
cloned = copy.deepcopy(loop)
print(cloned[0], cloned[1] is cloned)
print(cloned is loop, cloned[1] is loop)


1 True
False False


## When to use it

Mutate an argument when the function's job is to update the caller's object: fill a buffer, sort in place, register a listener. Rebind or return a new object when the caller must keep the old one. Document which one you did. A name like `sort` that returns `None` is the in-place contract; `sorted` returns a new list.

Use a shallow copy for a list or dict whose elements you will not mutate: numbers, strings, tuples of those. Use `deepcopy` when nested objects are mutable and the copy must be isolated. Do not `deepcopy` a structure that holds a file, a socket, or a thread lock — those refuse to be copied. Copy the data you own, not the resource.

For one level of rows that must be independent, `[row.copy() for row in grid]` is enough and cheaper than a full walk. Reach for `deepcopy` when you do not know the nesting, or when the graph has sharing you need to preserve as a copy of the shape.


## Common mistake

`list(grid)` looks like a private grid. Writing into a row updates the original. The outer lists compare as not the same object, so a check on the outer `is` stays green. The bug is one level down.


In [5]:
grid = [[0, 0], [1, 1]]
backup = list(grid)
backup[0].append(9)
print(grid)
print(backup)
print(grid is backup)
print(grid[0] is backup[0])


[[0, 0, 9], [1, 1]]
[[0, 0, 9], [1, 1]]
False
True


`grid` prints `[[0, 0, 9], [1, 1]]`. The outer containers differ, and the row does not. The fix copies the rows. `deepcopy` does that walk. After it, the append stays on `backup`.


In [6]:
grid = [[0, 0], [1, 1]]
backup = copy.deepcopy(grid)
backup[0].append(9)
print(grid)
print(backup)
print(grid[0] is backup[0])


[[0, 0], [1, 1]]
[[0, 0, 9], [1, 1]]
False


## Try it

Change `private_grid` so it returns `list(grid)` instead of a deep copy, and append `9` to `private[0]`. Which of `grid` and `private` shows `9`, and which expression copies each row without calling `deepcopy`?


## What this explains

- [Arguments](../../course/05-functions/02-arguments-lesson.ipynb): the parameter name receives a pointer. A default object, if it is mutable, is one object shared by every call that omits the argument.
- [Mutable defaults](../../course/10-pitfalls/01-mutable-defaults-lesson.ipynb): that shared default is call by sharing across calls, not a fresh list each time.
- [Shared rows](../../course/10-pitfalls/02-shared-rows-lesson.ipynb): multiplying one row repeats a pointer. The rows are aliases, the same fact as a shallow copy of a single inner list.
- [Objects, names, references](../01-objects/01-names-lesson.ipynb): assignment and `list()` from that lesson, now at the call boundary and one level deeper.


## Recap

- The callee shares the object and owns only its parameter slot. Mutation is visible. Rebinding is not.
- A shallow copy duplicates the container and aliases the contents. `deepcopy` duplicates the graph and stops on cycles.
- Copy as deep as the mutation you need to hide, and no deeper.

Next: [How list works](../03-lists/01-dynamic-array-lesson.ipynb).
